スペクトルデータにpeak|dipがある、無いenergy領域が存在するということは、
そのenergy領域の分散が大きい説明変数があるということです。
例えばPCAで同じpeak|dipがあるスペクトルは同じ説明変数の領域に集まり特徴抽出することができるはずです。
同様に、あるピークがあるDOSと同じピークがあるDOSとは距離が近く、異なるピークがあるDOSをとは距離が遠くなるmetricがあれば距離行列から特徴抽出することができるはずです。
また、
peakのenergy領域が全て同じわけでは無く、ピーク高さやピーク位置が少しずれることも多く、ある幅のenergy windowでsmearingを行うことで、良い特徴抽出を行ったことになる可能性もあります。

その後、クラスタリングにより、何かのmetricで距離が近いデータインスタンスをクラスター分割します。

これらを行ってみます。

以下では、
1. DR_TYPEで指定した次元圧縮を行う。
2. optional: SECOND_PREPROCESSで定義した次元圧縮を更に行う。
3. クラスタリング

を行っています。前処理をどう行うかでクラスタリング結果は大きく異なります。

ただし、表示用時には二次元に次元圧縮しています。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pymatgen.core.periodic_table import Element
import numpy as np
import os
import json

In [ ]:
# 第一段階
DR_TYPE =  "smearing" #  None|"smearing"|"pca"|"mds"|"tsne"
N_SMEARD_DOS = 10 # 2,3,5,10,20,30,40,50  # DR_TYPEによる次元圧縮後の次元を定義する。
_SMEAR_ALPHA = int(50*((N_SMEARD_DOS/5)**2)) # DF_TYPE=="smearing"の時に、SMEARINGのgaussian幅がmesh幅になるように調整している。

# 第二段階
SECOND_PREPROCESS = False # apply DSP_DR_TYPE after applying DF_TYPE dimentionality reduction.
DSP_DR_TYPE = "tsne" # pca|tsne|mds

FIGDIR="image_executed"
os.makedirs(FIGDIR, exist_ok=True)

In [ ]:
def get_data():
    filename = "../data/dos/hea4_dos.csv"
    df = pd.read_csv(filename, index_col=None)
    meta_names = ['Z1', 'Z2', 'Z3', 'Z4',
    'elm1', 'elm2', 'elm3', 'elm4', 'elements',  ]
    descriptor_names = []
    for i in range(100):
        descriptor_names.append("log10_dos{}".format(i+1))
    target_name = 'semicore'

    descriptor_names = descriptor_names[0:100]
    return df, descriptor_names, meta_names, target_name

g_df, g_logdos_names, g_meta_names, g_target_name = get_data()
g_descriptor_names = g_logdos_names
g_df

In [ ]:
def plot_x(df, descriptor_names, elm=None, xlabel="energy", ylabel="log10(DOS)", alpha=0.01):
    """plot X

    Args:
        df (pd.DataFrame): data
        descriptor_names ([str]]): a list of explanatory variables.
        elm (str, optional): element symbol to show in a plot. Defaults to None.
        xlabel (str, optional): name of x label. Defaults to "energy".
        ylabel (str, optional): name of y label. Defaults to "log10(DOS)".
        alpha (float, optional): alpha value of plot. Defaults to 0.01.
    """
    if elm is not None:
        dfq = df[df["elements"].str.contains(elm)]
        if dfq.shape[0]==0:
            print("elm={} 0".format(elm))
            return 
        Y = dfq[descriptor_names].values.T
    else:
        Y = df[descriptor_names].values.T
    n = Y.shape[1]
    x = list(range(len(descriptor_names)))
    
    fig, ax = plt.subplots()
    ax.plot(x,Y, alpha=alpha, c="black")
    if xlabel is not None:
        ax.set_xlabel(xlabel)
    if ylabel is not None:
        ax.set_ylabel(ylabel)
    if elm is not None:
        ax.set_title("{}, n={}".format(elm,n))
    else:
        ax.set_title("n={}".format(n))


In [ ]:
plot_x(g_df, g_descriptor_names)

In [ ]:
# yの表示
from collections import Counter
g_target_elm_list = np.unique(g_df[g_target_name].values)
print(len(g_target_elm_list),g_target_elm_list)
counter = Counter(g_df[g_target_name].values)
counter

In [ ]:
def smear_dos(v, newn, alpha, type_=1):
    """smear counter by 

    f(r), r_1,...r_M = sum_i^N exp(-alpha((r-i)^2))*v[i] if type_==1, i=1,N, M<N
    exp(-alpha((r-i)^2))*(r-i)^2 if type_==2,
    tanh(alpha((r-i)) if type_==3,
    
    この操作はNNの畳み込みとほぼ同じ操作です。

    Args:
        v (np.ndarray): values
        newn (int): new number of divisions
        alpha (float, optoinal): exp(-alpha*i*2). Defaults 1.0.

    Returns:
        np.ndarrays: smeared values
    """
    n = v.shape[1]

    i = [i for i in range(n)]
    i = np.array(i)/n
    
    r = np.linspace(0, 1, newn)

    r_i = r[np.newaxis, :]-i[:, np.newaxis]

    
    if type_ == 1:
        expr_r2 = np.exp(-alpha*r_i**2)
    elif type_ == 2:
        expr_r2 = np.exp(-alpha*r_i**2)*r_i**2
    elif type_ == 3:
        expr_r2 = np.tanh(alpha*r_i)
    else:
        raise ValueError("uknown type_={}".format(type_))

    expr_r2 = expr_r2[np.newaxis,:,:]
    
    v = v[:,:,np.newaxis]

    expr_r2_v = expr_r2 * v
    v2 = expr_r2_v.sum(axis=1)
    
    return v2, expr_r2


def add_convolution_variables(df, descriptor_names, n_new, alpha):
    """add convolution variables

    Args:
        df_obs (pd.DataFrame): データ。
        descriptor_names: 説明変数名リスト。
        n_new (int): 説明変数数。

    Returns：
        pd.DataFrame: データ
        [str]: descriptor_names: 
    """
    df = df.copy().reset_index(drop=True)
    X = df[descriptor_names].values

    v2, expr_r2 = smear_dos(df[descriptor_names].values, n_new, alpha=alpha)
    

    plt.plot(expr_r2[0,:,:])
    plt.title("expr_r2")
    
    smeared_names = []
    for i in range(n_new):
        smeared_names.append("smeared_dos{}".format(i))
        
    df_smeared = pd.DataFrame(v2, columns=smeared_names)
    return pd.concat([df,df_smeared], axis=1), smeared_names

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import MDS, TSNE

def dr_transform(df_sample, descriptor_names, dr_type="tsne", 
                  dr_prefix = "dr", ndim = 2,  random_state=1):
    """dimensionality reductionを行う。
    
    Args:
        df_sample (pd.DataFrame): データ.
        descriptor_names ([str]): df_sampleの説明変数名リスト.
        DR_TYPE (str): dimentionality reductionの名前。
        df_prefix (str, optional): dimentionality reductionしたカラム名のprefix. Defaults to "dr"
        ndim (int, optional):  dimentionality reductionの次元。Defaults to 2.
        seed (int, optional): random seed. Defaults to 2.
    
    Returns:
        pd.DataFrame: データ.
        [str]: dimentionality reductionしたカラム名リスト。
    """
    df_sample = df_sample.copy()
    X = df_sample[descriptor_names].astype(float).values
    print("apply",dr_type,"ndim=",ndim)
    if dr_type=="pca":
        dr = PCA(ndim)
    elif dr_type=="mds":
        dr = MDS(ndim, random_state= random_state)
    elif dr_type=="tsne":
        dr = TSNE(ndim,  init="pca", random_state=random_state)
    else:
        raise ValueError("unknown rd_type={}".format(dr_type))
        
    dr_names = []
    for i in range(ndim):
        dr_names.append("{}{}".format(dr_prefix, i))
        
    Xdr = dr.fit_transform(X)
    df_dr = pd.DataFrame(Xdr, columns=dr_names)
    df_sample = pd.concat([df_sample, df_dr],axis=1)
            
    return df_sample, dr_names

In [ ]:
if DR_TYPE is None:
    g_smearedlogdos_names = g_logdos_names
    g_descriptor_names = g_smearedlogdos_names
elif DR_TYPE=="smearing":
    g_df, g_smearedlogdos_names = add_convolution_variables(
        g_df, g_logdos_names, N_SMEARD_DOS, _SMEAR_ALPHA)
    #sns.pairplot(g_df[g_descriptor_names])
    g_descriptor_names = g_smearedlogdos_names
elif DR_TYPE == "pca" or DR_TYPE=="mds":
    g_df, g_smearedlogdos_names = dr_transform(
        g_df, g_logdos_names, dr_prefix=DR_TYPE, dr_type= DR_TYPE, ndim=N_SMEARD_DOS)
    g_descriptor_names = g_smearedlogdos_names
elif DR_TYPE == "tsne":
    if N_SMEARD_DOS<=5:
        g_df, g_smearedlogdos_names = dr_transform(
            g_df, g_logdos_names, dr_prefix=DR_TYPE, dr_type= DR_TYPE, ndim=N_SMEARD_DOS)
        g_descriptor_names = g_smearedlogdos_names   
    else:
        raise ValueError("It takes much time. DR_TYPE={} isn't supported.".format(DR_TYPE))
else:
    raise ValueError("unknown DR_TYPE={}".format(DR_TYPE))

In [ ]:
# 図示するための二次元座標の用意。
# g_df[g_dr2_names]は二次元図示のためだけに用いる。
g_df, g_dr2_names = dr_transform(g_df, g_smearedlogdos_names,
                                        dr_type=DSP_DR_TYPE, dr_prefix = "dr2")

In [ ]:
from sklearn.cluster import KMeans
from  sklearn.mixture import GaussianMixture

def df_clustering(df, descriptor_names, n_clusters, cluster_name="kmeans", random_state=1):
    """クラスリングをして結果を表示する。
    
    KMeans(n_clusters, n_init='auto',...) n_init is added to avoid printing a FutureWarning message.
    
    Args:
        df (pd.DataFrame): データ。
        descriptor_names ([str]): 説明変数名リスト。
        n_clusters (int): クラスタ数。
        
    Returns:
        pd.DataFrame: データ。
    """
    df = df.copy()
    kmeans = KMeans(n_clusters, n_init='auto', random_state=random_state)
    X = df[descriptor_names].values
    kmeans.fit(X)
    yp = kmeans.predict(X)
    df[cluster_name] = yp
    return df, cluster_name

In [ ]:
g_n_clusters=len(g_target_elm_list)
if not SECOND_PREPROCESS :
    g_df, g_cluster_name = df_clustering(g_df, g_descriptor_names, n_clusters=g_n_clusters)
else:
    g_df, g_cluster_name = df_clustering(g_df, g_dr2_names, n_clusters=g_n_clusters) # smearingしてからt-sneする。

In [ ]:
def plot_cluster(df, target_name, dr_names, ax,
                  alpha=1.0, s=5,         ):
    """クラスリングをした結果を表示する。

    Args:
        df (pd.DataFrame): data
        target_name (str): target variable name.
        dr_names ([str]]): a list of exaplanatory variables.
        ax (Axes): matplotlib axis.
        alpha (float, optional): alpha value of plot. Defaults to 1.0.
        s (int, optional): point size. Defaults to 5.
    """
    yp = df[target_name]
    X_dr = df[dr_names].values
    uniq_yp = np.unique(yp)
    n_clusters = uniq_yp.shape[0]
    
    
    for i in uniq_yp:
        idx = yp==i
        ax.scatter(X_dr[idx,0], X_dr[idx,1], alpha=alpha, s=s, label=str(i))  # ラベルを追加
        center = (X_dr[idx,0].mean(), X_dr[idx,1].mean())
        ax.text(center[0], center[1], str(i), fontsize=20)
        ax.scatter([center[0]], [center[1]], marker="x", color="black")
    ax.set_title("{}, {}".format(target_name, n_clusters))
    # ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0,)
    

def plot_clusters(df, target_names,  dr2_names,  alpha=1.0, s=5):
    """クラスタを可視化表示する。

    Args:
        df (pd.DataFrame): data
        target_names ([str]): a list of target names.
        dr2_names ([str]): explanatory variables.
        alpha (float, optional): alpha channel value of ploot. Defaults to 1.0.
        s (int, optional): point size. Defaults to 5.
    """
    fig, axes = plt.subplots(1,len(target_names),figsize=(10,5))
    for name,ax in zip(target_names, axes):
        plot_cluster(df, name, dr2_names, ax, alpha, s)
    figfilename =  "cluster_{}.png".format("_".join([DR_TYPE,
                                str(N_SMEARD_DOS),
                               str(_SMEAR_ALPHA),str(SECOND_PREPROCESS),
                               DSP_DR_TYPE]))
    figpath = os.path.join(FIGDIR,figfilename)
    fig.savefig(figpath)
    print("save to", figpath)
        
plot_clusters(g_df, [g_target_name,g_cluster_name],  g_dr2_names)

上図のMo, Hgの分離が最も問題になります。

smearnigしてから二次元tsneを行った変数が最も分離度が良さそうです。
このscriptではSECOND_PREPROCESS=Trueとするとsmearnigしてから二次元tsne後の変数のクラスタリングを行います。

In [ ]:
"""from stack overflow
https://stackoverflow.com/questions/41540751/sklearn-kmeans-equivalent-of-elbow-method
"""
def elblow_method(X, ncluster_max = 10, random_state=1):
    """eblow法

    Args:
        X (np.ndarray)): 説明変数
        ncluster_max (int, optional): 最大クラスタ多数. Defaults to 10.
        random_state (int, optional): KMeansのrandom_state. Defaults to 1.
    """
    result = []

    ncluster_list = range(1,ncluster_max)
    for i  in ncluster_list:               # 1~ncluster_maxクラスタまで一計算 
        km = KMeans(n_clusters=i,n_init='auto',
                    random_state=random_state)
        km.fit(X)                         # クラスタリングの計算を実行
        result.append(km.inertia_)   # km.fitするとkm.inertia_が得られる

    plt.plot(ncluster_list, result,marker='o')
    plt.xlabel('nclusters')
    plt.ylabel('Distortion')
    plt.title('Elbow curve')

    plt.show()
    
g_X = g_df[g_descriptor_names].values
elblow_method(g_X, ncluster_max=len(g_target_elm_list)+5)

In [ ]:
g_df

In [ ]:
def plot_dos_cluster(df, cluster_name, smearedlogdos_names, logdos_names, ):
    """
    plot DOS of clusters
    
    Args:
        df (pd.DataFrame): データ.
        cluster_name (str): クラスターカラム名。
        smearedlogdos_names ([str]): 次元圧縮した説明変数名.
        logdos_names ([str]): 説明変数名.
    """
    
    uniq_clusters = np.unique(df[cluster_name].values)
    n_clusters = uniq_clusters.shape[0]
    fig, axes = plt.subplots(n_clusters,2, figsize=(5,2.5*n_clusters))

    for i in uniq_clusters:
        df_select = df[df[cluster_name]==i]
        if df_select.shape[0]>0:

            ax = axes[i,0]
            y = df_select[smearedlogdos_names].values.T
            x = np.array(list(range(y.shape[0])))
            ax.plot(x,y, alpha=0.1, c="black")
            ax.set_title("{}, n={}".format(i, y.shape[1]))

            ax = axes[i,1]
            y2 = df_select[logdos_names].values.T
            x2 = np.array(list(range(y2.shape[0])))
            ax.plot(x2,y2, alpha=0.1, c="black")

    fig.tight_layout()
        
plot_dos_cluster(g_df,  g_cluster_name, g_smearedlogdos_names, g_logdos_names, )

In [ ]:
g_df[g_df[g_cluster_name]==0]["elements"]
#となっている。最頻出共通要素(元素名)を得る。

以下のコードを動かすにはmlxtendのインストールが必要です。
インストールコマンド例
```
pip install mlxtend
```

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

def get_freq_items(df_sample, idx, transacion_names, predictedtarget_name, 
                   elements, min_support=0.4, min_nitem=1):
    """頻出マイニングを行う。
    
    該当するdf_sample.loc[idx, predictedtarget_name]がinplaceで加えられるので、
    予めdf_sample[predictedtarget_name]列を作ること。

    Args:
        transaction ([[str]]): transaction.
        min_support (float, optional): minimum value of support ratio. Defaults to 0.4.

    Returns:
        pd.DataFrame: データ。
    """
    # min_threshold = 0.8
    transaction = df_sample.loc[idx, transacion_names].astype(str).values
    
    te = TransactionEncoder()
    te.fit(transaction)
    te_ary = te.fit(transaction).transform(transaction)
    df = pd.DataFrame(te_ary, columns=te.columns_)
    df_freq_items = apriori(df, min_support=min_support,
                            max_len=10000, use_colnames=True, verbose=1)
    nlist = []
    for item in df_freq_items["itemsets"]:
        nlist.append(len(item))
    df_freq_items["nitem"] = nlist
    df_freq_items = df_freq_items[df_freq_items["nitem"]>=min_nitem]
    
    df_freq_items.sort_values(by="support", ascending=False, inplace=True)
    
    flag = False
    for items in df_freq_items["itemsets"].values:
        for item in items:
            if item in elements:
                df_sample.loc[idx,predictedtarget_name] = item
                print("item", item,"is set.")
                flag = True
                break
        if flag:
            break
    
    return df_freq_items

後出の7100で示しますが、itemset miningにより最も頻度が大きい共通itemを得ることができます。

In [ ]:
def add_predictedname(df, cluster_name, target_elm_list, 
                      predictedtarget_name= "semicorep", 
                      elms=['elm1','elm2', 'elm3', 'elm4']):
    """
    cluster id は番号なので、名前と合わせられない。
    頻出マイニングで番号と名前を合わせる。
    
    df[predictedtarget_name]がinplaceで加えられる。
    
    Args:
        df (pd.DataFrame): データ。
        cluster_name (str): 予測クラスターのカラム名。
        target_elm_list (str): 目的変数名のユニークリスト。
        predictedtarget_name (str, optional): 名前にしたクラスターのカラム名
        elms ([str]): transactionに用いるカラム名リスト。
        
    
    """
    df[predictedtarget_name] = None
    transacion_names = elms
    transacion_names.append(cluster_name)
    uniq_clusters = np.unique(df[cluster_name].values)
    for i in uniq_clusters:
        print("-------")
        print("cluster",i)
        display(get_freq_items(df, df[cluster_name]==i, transacion_names, predictedtarget_name,
               target_elm_list, min_support=0.4, min_nitem=2))
g_predictedtarget_name = "semicorep"
add_predictedname(g_df, g_cluster_name, g_target_elm_list, g_predictedtarget_name)

In [ ]:
from sklearn.metrics import confusion_matrix

In [ ]:
def make_cm_save(df, target_name, predictedtarget_name, target_elm_list, dr_type, n_dr_dim):
    """
    形式的にconfusion matrixを作りセーブする。
    
    Args:
        df (pd.DataFrame): データ。
        target_name (str): 観測値目的変数名カラム。
        predictedtarget_name (str): 予測値目的変数名カラム。
        target_elm_list ([str]): 観測値のユニークリスト。
        dr_type (str): 次元圧縮手法名。
        n_dr_dim (int): 次元圧縮後の次元。
        
    Returns:
        csvファイル名。
    """

    cm = confusion_matrix(df[target_name],df[predictedtarget_name])
    # 上のcluster番号と名前の対応がつかなかった場合はエラーが起きる。
    
    df_cm = pd.DataFrame(cm, index=target_elm_list,columns=target_elm_list)
    display(df_cm)
    
    OUTPUTDIR="data_executed"
    os.makedirs(OUTPUTDIR, exist_ok=True)
    filename = "hea4_y_yp_{}_{}.csv".format(dr_type, n_dr_dim)
    filepath = os.path.join(OUTPUTDIR,
                              filename)
    df[[target_name,predictedtarget_name]].to_csv(filepath, index=False)
    print("saved to",filepath)
    return filepath

if not SECOND_PREPROCESS:

    if DR_TYPE is None:
        g_filepath = make_cm_save(g_df, g_target_name, g_predictedtarget_name, g_target_elm_list, 
                                  "None", len(g_logdos_names))
    else:
        g_filepath = make_cm_save(g_df, g_target_name, g_predictedtarget_name, g_target_elm_list, 
                                  DR_TYPE, N_SMEARD_DOS)
else:
    g_filepath = make_cm_save(g_df, g_target_name, g_predictedtarget_name, g_target_elm_list, 
                                  "smearing-{}-2".format(DSP_DR_TYPE), N_SMEARD_DOS)

In [ ]:
pd.read_csv(g_filepath)

In [ ]:
DR_TYPE, N_SMEARD_DOS,"done"

結果は乱数により大きく変わります。


# 形式的な分類性能

クラスタリングですが、labelがあるので形式的に分類性能を評価できます。

In [ ]:
sorted(np.unique(g_df[g_target_name]))

In [ ]:
sorted(np.unique(g_df[g_predictedtarget_name]))

In [ ]:
from sklearn.metrics import classification_report
msg = classification_report(g_df[g_target_name],g_df[g_predictedtarget_name]) # 文字列を返す。
print(msg)

上のセルは予測値が全てのラベルを持っていないとwarningが出ます。

## 問題

同じデータでclassificationを行う。


### 回答

- 1150.1150 ソース
- 1150.1200 summary